### Step 1: Preparing web scraper.

In [43]:
# url = 'https://www.adexchanger.com/platforms/pinterest-owned-tvscientific-gets-closer-to-the-bidstream-with-openxs-new-api-suite/'

In [44]:
from bs4 import BeautifulSoup
import requests
from IPython.display import Markdown, display

In [45]:
# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 20,000 characters
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input", "svg", "iframe", "noscript"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:20000]

In [46]:
# content = fetch_website_contents(url)
# print(content)

### Step 2: Preparing system & user prompt to extract business names

In [47]:
system_prompt = """You are a business name extraction specialist. Your task is to carefully read the provided article and identify all business names mentioned within it.

INSTRUCTIONS:
1. Read the entire article thoroughly
2. Identify all business names, including:
   - Company names (e.g., "Microsoft Corporation", "Apple Inc.")
   - Brand names that represent businesses (e.g., "Nike", "Coca-Cola")
   - Subsidiaries and divisions when explicitly named as business entities
   - Startups and small businesses
   - Both formal legal names and commonly used trade names

3. DO NOT include:
   - Product names that aren't business names (e.g., "iPhone" unless referring to the business)
   - Person names
   - General industry terms or categories
   - Government agencies or non-profit organizations (unless the context clearly indicates they operate as businesses)

4. Return your findings as a bullet point list using this format:
   - Business Name 1
   - Business Name 2
   - Business Name 3

FORMATTING RULES:
- Use the exact spelling and capitalization as it appears in the article
- Include any legal suffixes (Inc., LLC, Corp., Ltd.) if mentioned
- One business name per bullet point
- If a business is mentioned multiple times, list it only once
- Sort alphabetically for easier reading
- If no business names are found, respond with: "No business names identified in this article."

OUTPUT ONLY:
Provide only the bullet point list with no additional commentary, explanations, or preamble.
"""

In [48]:
def get_business_names_user_prompt(url):
    user_prompt = """
    Please extract all business names from the following article:
    """

    content = fetch_website_contents(url)
    user_prompt += "".join(content)
    return user_prompt


In [49]:
# get_business_names_user_prompt(url)

### Step 3: Making a call to OpenAI LLM to get business names from the article

In [50]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [51]:
def extract_business_name(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role" : "system", "content" : system_prompt},
            {"role" : "user", "content" : get_business_names_user_prompt(url)}
        ]
    )
    businesses = response.choices[0].message.content
    #display(Markdown(businesses))
    return businesses

In [52]:
# extract_business_name(url)

### Step 4: Preparing system and user prompt to categorise businesses

In [53]:
business_classification_system_prompt = """
You are an expert ad tech analyst specializing in ecosystem classification and market positioning. Your role is to analyze businesses and provide comprehensive, accurate classifications within the advertising technology landscape.

## Classification Framework

When analyzing a business, provide the following structured analysis:

### 1. Primary Category
Identify the core business model from these categories:
- **DSP (Demand-Side Platform)**: Platforms enabling advertisers/agencies to buy ad inventory programmatically
- **SSP (Supply-Side Platform)**: Platforms enabling publishers to sell ad inventory programmatically
- **Ad Exchange**: Marketplaces connecting buyers and sellers for real-time bidding
- **Agency Holding Company**: Parent companies owning multiple agencies and ad tech assets
- **Ad Network**: Aggregators of publisher inventory selling to advertisers
- **Data Provider**: Companies providing audience data, segments, or identity solutions
- **Retail Media Network**: Retailer-owned platforms monetizing first-party shopper data
- **Ad Server**: Technology delivering and tracking ad content
- **Verification/Measurement**: Tools for ad fraud detection, viewability, brand safety, and attribution
- **Creative Technology**: Dynamic creative optimization (DCO) and creative management platforms
- **Customer Data Platform (CDP)**: Systems unifying customer data across touchpoints
- **DMP (Data Management Platform)**: Platforms for collecting and managing audience data
- **CTV/OTT Platform**: Connected TV and over-the-top streaming advertising solutions
- **Marketing Cloud/Suite**: Integrated marketing technology stacks
- **Other**: Specify if the business doesn't fit standard categories

### 2. Secondary Categories
List any additional categories where the business operates significantly. Many ad tech companies are multi-product platforms operating across categories.

### 3. Ecosystem Position
Classify the business's position in the value chain:
- **Demand Side/Buy Side**: Serving advertisers, agencies, and media buyers
- **Supply Side/Sell Side**: Serving publishers, content creators, and media sellers
- **Infrastructure**: Enabling transactions, data flow, or technical operations for both sides
- **Data/Measurement**: Providing data assets or measurement capabilities that span the ecosystem

Note: Some businesses occupy multiple positions (e.g., "Demand Side + Infrastructure")

### 4. Market Segment
Identify the primary market focus:
- **Enterprise**: Serving large advertisers, agencies, or publishers
- **SMB (Small-Medium Business)**: Serving smaller advertisers or publishers
- **Self-Service**: Platforms with automated, low-touch onboarding
- **Managed Service**: High-touch, service-intensive offerings
- **Vertical-Specific**: Focused on particular industries (e.g., automotive, retail, travel)
- **Format-Specific**: Specialized by ad format (e.g., video, mobile, native, CTV)
- **Geographic**: Region-specific operations (e.g., APAC-focused, US-only)

## Analysis Approach

1. **Research thoroughly**: Consider the company's website, product descriptions, case studies, and public materials
2. **Look beyond marketing**: Distinguish between aspirational positioning and actual technical capabilities
3. **Consider evolution**: Note if the company has pivoted or expanded categories over time
4. **Identify core revenue**: Focus on what generates the majority of revenue, not tangential features
5. **Acknowledge complexity**: Many modern ad tech companies are multi-product platforms; classify accurately across multiple dimensions
6. **Note ownership**: Parent company relationships can affect classification (e.g., agency-owned tech vs. independent)

## Output Format

Present your analysis clearly:

**Primary Category**: [Category]
**Secondary Categories**: [List if applicable, or "None"]
**Ecosystem Position**: [Position(s)]
**Market Segment**: [Segment(s)]

**Brief Justification**: [2-3 sentences explaining the classification based on business model, product offerings, and market position]

## Important Notes

- Ad tech companies often blur category lines; prioritize accuracy over forcing single-category classifications
- The distinction between DSPs and ad networks has blurred as networks added programmatic capabilities
- Many "platforms" are actually suites containing multiple category products
- Be specific about what makes a company fit a category (e.g., real-time bidding for DSP, header bidding for SSP)
"""

In [54]:
def business_classification_user_prompt(url):
    user_prompt = """
    Classify these businesses
    """
    businesses = extract_business_name(url)
    user_prompt += "".join(businesses)
    return user_prompt
    

In [55]:
# print(business_classification_user_prompt(url))

In [56]:
def classify_business(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role" : "system", "content" : business_classification_system_prompt},
            {"role" : "user", "content" : business_classification_user_prompt(url)}
        ]
    )
    classification = response.choices[0].message.content
    display(Markdown(classification))
    #return classification

In [57]:
# print(classify_business(url))

In [58]:
print(classify_business(input()))

Here’s a best-effort classification for each entity, using the provided taxonomy. For many legacy media brands, the primary category is not a core ad-tech product, so I treat them as publisher/media or general data/media companies with ad-tech relevance.

- AdExchanger
  Primary Category: Other
  Secondary Categories: None
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: AdExchanger is a trade publication focused on ad tech and digital marketing, not a programmatic buying or selling platform. It serves the industry with news and insights rather than delivering ad tech software.

- Bloomberg
  Primary Category: Other
  Secondary Categories: Data Provider
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: Bloomberg is a large data and media company (financial data, analytics, news) rather than a pure ad-tech platform. It provides data assets and market information, which places it in the Data/Measurement space more than DSP/SSP roles.

- Business Insider
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: A major publisher and digital media brand; primarily monetizes via media properties. It’s not primarily an ad-tech platform, though it has advertising business and data assets.

- ChatGPT
  Primary Category: Other
  Secondary Categories: None
  Ecosystem Position: Infrastructure
  Market Segment: Self-Service
  Brief Justification: An AI platform used broadly to build capabilities (not an ad-tech buyer/seller, DSP/SSP, or data provider). It sits as infrastructure for developers and products.

- Conductor
  Primary Category: Marketing Cloud/Suite
  Secondary Categories: Data Provider
  Ecosystem Position: Infrastructure
  Market Segment: Enterprise
  Brief Justification: Conductor provides digital marketing orchestration/SEO and content marketing tooling, i.e., a marketing cloud/suite with data insights for enterprise teams.

- Digiday
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: Digiday is a trade publication covering media/advertising tech; it is not a programmatic platform but a media/insights publisher.

- Facebook
  Primary Category: DSP
  Secondary Categories: Data Provider
  Ecosystem Position: Demand Side + Infrastructure
  Market Segment: Self-Service + Enterprise
  Brief Justification: Facebook Ads (and Meta’s ad tech stack) operates as a demand-side platform with extensive data assets; it also provides ad-serving infrastructure and audience data.

- Google
  Primary Category: DSP
  Secondary Categories: Ad Server; Data Provider
  Ecosystem Position: Demand Side + Infrastructure
  Market Segment: Enterprise + Self-Service
  Brief Justification: Google runs DV360 (DSP), Google Ad Manager (ad server), and has substantial data assets; it spans demand-side and infrastructure roles in ad-tech.

- HuffPost
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: A major publisher with advertising business; not a stand-alone ad-tech product, but a media asset with data/monetization capabilities.

- NBCU (NBCUniversal)
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: A large media company with several ad-tech initiatives, but primarily a content publisher with monetization and data capabilities.

- OpenX
  Primary Category: SSP
  Secondary Categories: Ad Exchange
  Ecosystem Position: Supply Side + Infrastructure
  Market Segment: Enterprise
  Brief Justification: OpenX is a core sell-side platform, operating as an SSP and exchange for publishers and the programmatic market.

- People Inc.
  Primary Category: Other
  Secondary Categories: None
  Ecosystem Position: Other
  Market Segment: Enterprise
  Brief Justification: Without a clear ad-tech product, classified as a non-core business; not a known DSP/SSP/infrastructure play.

- Pew Research
  Primary Category: Data Provider
  Secondary Categories: Data/Measurement
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: A research/think-tank organization providing data and public opinion metrics; serves as a data asset in the ecosystem.

- Press Gazette
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: Trade/press publication; primarily media and industry coverage rather than a programmatic ad-tech product.

- Reddit
  Primary Category: SSP
  Secondary Categories: Ad Exchange
  Ecosystem Position: Supply Side + Infrastructure
  Market Segment: Self-Service + Enterprise
  Brief Justification: Reddit monetizes its own inventory and offers programmatic access via exchanges; functions as a sell-side platform and marketplace edge.

- Seer Interactive
  Primary Category: Other
  Secondary Categories: Managed Service
  Ecosystem Position: Demand Side
  Market Segment: Managed Service
  Brief Justification: An independent digital marketing agency specializing in analytics/SEM/SEO; operates on the buy side as an agency service provider.

- Semrush
  Primary Category: Marketing Cloud/Suite
  Secondary Categories: Data Provider
  Ecosystem Position: Infrastructure
  Market Segment: Self-Service
  Brief Justification: A marketing analytics/SEO toolkit with data assets; delivered via subscription/self-service.

- Similarweb
  Primary Category: Data Provider
  Secondary Categories: Marketing Cloud/Suite
  Ecosystem Position: Data/Measurement
  Market Segment: Self-Service
  Brief Justification: Provides competitive intelligence and website analytics data; acts as a data asset in the ecosystem.

- Stereogum
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: A music/podcasts site; primarily media/publisher, not a programmatic ad-tech product.

- TechRadar
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: Tech news site; media/publisher with potential ad-tech data/monetization discussions, not a core ad-tech platform.

- The Mail Online
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: Major online publisher; monetizes via ads and data assets, not a standalone ad-tech platform.

- The New York Times
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: Large publisher with significant digital ad monetization and data assets; not primarily a standalone DSP/SSP.

- The Wall Street Journal
  Primary Category: Other
  Secondary Categories: Publisher/Media
  Ecosystem Position: Data/Measurement
  Market Segment: Enterprise
  Brief Justification: Major publisher with advertising and data assets; main business is media, not a pure ad-tech platform.

- X (formerly Twitter)
  Primary Category: DSP
  Secondary Categories: Data Provider
  Ecosystem Position: Demand Side + Infrastructure
  Market Segment: Self-Service + Enterprise
  Brief Justification: X Ads represents a self-serve and enterprise ad platform; data assets and audience targeting accompany the buying/selling infrastructure.

Notes and guidance:
- Many incumbents are primarily publishers or data/content platforms; in those cases I classify them as “Other” with a secondary leaning toward Publisher/Media or Data/Measurement to reflect their real-world role in advertising ecosystems.
- OpenX is a clear SSP and exchange leader; Reddit and X are placed to reflect their ad inventory and programmatic components.
- Some brands (e.g., Google, Facebook, X) span multiple roles in practice (advertising platform, data provider, and infrastructure)—the classifications above choose the dominant/most relevant primary category while noting secondary facets.

If you want a stricter mapping for a subset (e.g., only core ad-tech products and no media properties), I can reclassify those accordingly.

None
